# Homework 1: Feed-forward Neural Networks (100 points)

### Overview

Below you will find a PyTorch implementation of a feed-forward neural network for image recognition. We use the popular MNIST dataset, where the model predicts a single digit (0-9) for a black-and-white photo of a handwritten digit. This is a _classification_ task.

### NN Architecture

Each image has size 28x28 grayscale pixel values between 0 and 255. In preprocessing, we flatten each image to a single vector of length $28^2 = 784$, which serves as the entire input for the model.

For each image, we aim to predict one of ten classes (0-9). We could use an output layer $y$ of size 1 (a single neuron) -- for example, using a naive mapping like prediction $p = \mathrm{int}(10y)$. But this presupposes that a handwritten 0 is similar to a handwritten 1 and very different than a handwritten 9, which isn't the case. So instead we use an output layer $y$ of size 10, where the prediction $p = argmax(y)$, so each output neuron controls the likelihood for a particular class.

We use a simple two-layer neural network. To begin, we will have an input size of 784, a hidden layer of size 5, and an output layer of size 10.

### Your Task

At the bottom of this notebook file, there are a series of questions testing your understanding of this neural network architecture. Some questions include instructions where you will need to modify hyperparameters (notated in the code below) and re-run the model to investigate the changed results. __There is no need to read through the following code in depth to answer the questions, but it may be useful as a reference.__

Below each question is a cell with the text “Type Markdown and LaTex.” Double-click the cell and type your response to the question. Save your responses by clicking on the floppy disk icon or choosing File - Save and Checkpoint.

After responding to the questions, click the submit button at the top of the notebook to submit your assignment for grading.

In [1]:
import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms

# set a manual seed so results are reproducible across runs
torch.manual_seed(0)

In [2]:
from mads.lib.path import assets
root_dir = assets.find("week1")

# load the MNIST training and test datasets
# ToTensor() converts each image to a tensor and scales pixel values to [0, 1]
trainDataset = datasets.MNIST(root=root_dir, train=True, transform=transforms.ToTensor(), download=True)
testDataset = datasets.MNIST(root=root_dir, train=False, transform=transforms.ToTensor())

In [3]:
class NNModel(nn.Module):
    def __init__(self, inputSize, outputSize, hiddenSize, activate):
        super().__init__()

        # select the requested activated function
        self.activate = nn.Sigmoid() if activate == "Sigmoid" else nn.Tanh() if activate == "Tanh" else nn.ReLU()

        # define a simple two-layer fullt connected neural network
        self.layer1 = nn.Linear(inputSize, hiddenSize)
        self.layer2 = nn.Linear(hiddenSize, outputSize)
        
    def forward(self, X):
        # pass inputs through the hidden layer, activation, and output layer
        hidden = self.activate(self.layer1(X))
        return self.layer2(hidden)

In [6]:
# The dimensionality of the input
inputSize = 784
# Number of neurons in the first layer
hiddenSize = 300
# Number of neurons in the second layer
outputSize = 10
# Activation function (default: ReLU, options: Sigmoid, Tanh, ReLU)
activation = "ReLU"
# Learning rate
learningRate = 0.001
# Number of training epochs
numEpochs = 5
# Number of training examples per batch
batchSize = 200

In [ ]:
# create data loaders for mini-match training and evaluation
trainLoader = torch.utils.data.DataLoader(dataset=trainDataset, batch_size=batchSize, shuffle=True)
testLoader = torch.utils.data.DataLoader(dataset=testDataset, batch_size=batchSize, shuffle=False)

# initialize the model, loss function, and optimizer
net = NNModel(inputSize, outputSize, hiddenSize, activation)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=learningRate)

print('>>> Beginning training!')
for epoch in range(numEpochs):
    for i, (images, labels) in enumerate(trainLoader):
        # flatten each 28x28 image into a length-784 vector
        images = images.view(-1, 28*28)

        # clear old gradients from the previous optimization step
        optimizer.zero_grad()
        
        # Forward propagation: compute class scores
        outputs = net(images)
        
        # compute loss and backpropagate gradients
        loss = criterion(outputs, labels)
        loss.backward()
        
        # update model parameters
        optimizer.step()
        
        # display progress every 100 mini-batches
        if (i+1) % 100 == 0:
            print('Epoch [{}/{}], Step [{}/{}], Loss: {}'.format(epoch+1, numEpochs, i+1, len(trainDataset)//batchSize, loss))

print()
print('>>> Beginning validation!')
correct, total = 0, 0
for i, (images, labels) in enumerate(testLoader):
    images = images.view(-1, 28*28)

    # get predicted class from the output logits
    outputs = net(images)
    _, prediction = torch.max(outputs, axis=1)
    correct += torch.sum(prediction == labels)
    total += labels.size(0)

# report validation accuracy as a percentage
print('Validation accuracy: {}%'.format(correct.item()/total*100))

## Homework Questions

Your goal is to improve the model's accuracy by tuning hyperparameters. If a question asks you to modify a hyperparameter and you obtain improved results, retain that hyperparameter change for subsequent questions. Otherwise, revert back to the original hyperparameter value.

**To make sure your code produces consistent results, it is advisable to click "Kernel -> Restart & Run All" every time you want to run your code.**

### Question 1: Loss Minimization & Gradient Descent (5 points)

Given a neural network with model parameters $\theta$, loss function $E$, and learning rate $\alpha$, what is the correct method to perform gradient descent?

a) $\theta_i += \alpha E$

b) $\theta_i -= \alpha E$

c) $\theta_i += \alpha\frac{\partial E}{\partial \theta_i}$

d) $\theta_i -= \alpha\frac{\partial E}{\partial \theta_i}$

---
**Answer:** d) $\theta_i -= \alpha\frac{\partial E}{\partial \theta_i}$

---

### Question 2: Class Imbalance (10 points)

Imagine you are an engineer tasked with helping a company to identify faulty parts early using an machine learning-based image recognition system. What evaluation metric would you use? More specifically, explain why a raw percent accuracy score would be a poor choice of evaluation metric for this problem space.

---
**Answer:** Raw accuracy can be misleading in imbalanced problems because it rewards the model for getting the majority class right, even if it completely failts on the rare cases. In this setting, the rare faulty parts are the cases to focus on. Metrics like *recall* help measure how many faulty parts are caught, *precision* helps measure how many predicted faults are truly faulty, and *F1-score* balances both. These metrics are likely better evaluation metrics to use in this situation. 

---

### Question 3a:  Size of a Hidden Layer (10 points)

Explain how the hidden layer size influences the architecture of a feed-forward neural network. In doing so, note what can happen if the hidden size is too large and what can happen if the hidden size is too small.

---
**Answer:** Hidden size controls model complexity. Too small and the network cannot represent the task well. Too large and it may overfit or require more training data and computation.

---

### Question 3b: Size of a Hidden Layer  (10 points)

Increase the hidden size from 5 to 300 and re-run your trial. How does the accuracy change?

_a) It increases, since the model learns more quickly_

_b) It increases, since the model has more memory and can learn more complex features_

_c) It decreases, since the model has to learn more parameters and it doesn't have enough time_

_d) It decreases, since the model has less memory_

---
**Answer:** 

b) It increases, since the model has more memory and can learn more complex features. When I increase the hidden size from 5 to 300, the model has much greater capacity to learn patterns in the MNIST images. A hidden size of 5 is too small for a task with this much visual variability, so increasing it improves the model's ability to represent useful features and increases accuracy.

---

### Question 4a: Learning Rate  (10 points)

Explain the purpose of a learning rate. In doing so, note what can happen if the learning rate is too large and what can happen if the learning rate is too small.

---
**Answer:** The learning rate controls how large each parameter update is during optimization. It affects both speed and stability of training. If the learning rate is too large, the updates can overshoot the minimum of the loss function, causing unstable training or preventing convergence entirely. If the learning rate is too small, training becomes slow and the model may not improve much within the available number of epochs.
Validation accuracy: 95.39999999999999%

---

### Question 4b: Learning Rate  (10 points)

Increase the learning rate from 0.001 to 1. How does the accuracy change?

a) It increases, since the model learns more quickly

b) It increases, since the model is better able to converge

c) It decreases, since the model learns too slowly

d) It decreases, since the model is not able to converge

---
**Answer:**

d) It decreases, since the model is not able to converge. 
The update steps become too large and training becomes unstable.

---

### Question 5a: Activation Functions (10 points)

Explain the main purpose of an activation function in neural networks. Also, explain the main benefit of the Tanh activation function over the Sigmoid activation function, and the main benefit of the ReLU activation function over the Sigmoid activation function.

---
**Answer:** 

- The main purpose of an *activation function* is to introduce nonlinearity into a neural network. Without activation functions, a network made of multiple linear layers would still behave like a single linear model and would not be able to learn complex nonlinear patterns.

- The main benefit of *Tanh over Sigmoid* is that Tanh outputs values between -1 and 1, so it is zero-centered. This often makes optimization easier than Sigmoid, which outputs values between 0 and 1 and can cause less balanced gradients.

- The main benefit of *ReLU over Sigmoid* is that ReLU helps reduce the vanishing gradient problem and is computationally simple. For positive inputs, its gradient does not shrink the way Sigmoid does, so training is often faster and more effective.

---


### Question 5b: Activation Functions (5 points)

Change the activation function in the hyperparameter list above to determine which activation function is most effective at this task.

a) ReLU: learning rate = 0.001, hidden size = 300, Validation accuracy: 97.52%

b) Sigmoid: learning rate = 0.001, hidden size = 300, Validation accuracy: 95.39999999999999%

c) Tanh: learning rate = 0.001, hidden size = 300, Validation accuracy: 97.0%

---
**Answer:** 

a) *ReLU* is the most effective activation function for this task. Compared with Sigmoid and Tanh, it typically trains faster and avoids much of the vanishing gradient problem. On MNIST, this usually leads to better validation accuracy.

---

### Question 6: Overfitting  (10 points)

Define overfitting and explain how it can damage model training and results.

---
**Answer:** *Overfitting* occurs when a model learns the training data too closely, including noise or accidental details that do not generalize to new data. An overfit model usually has very high training performance but noticeably worse validation or test performance.

This damages results because the goal of machine learning is to generalize well to *unseen* examples, not just memorize the training set. A model that overfits may appear strong during training but *perform poorly* when deployed on *real-world data*.

---

### Question 7: Early Stopping  (10 points)

Outline a procedure for early stopping to prevent overfitting. Clearly describe how you’d use the training, validation, and test sets accuracy to decide when to stop.

---
**Answer:**

A common early stopping procedure is:

1. Split the data into training, validation, and test sets.
2. Train the model using the training set.
3. After each epoch, evaluate the model on the validation set.
4. Save the model whenever the validation accuracy improves (or validation loss decreases).
5. If the validation performance stops improving for a chosen number of epochs, stop training.
6. After training is complete, evaluate the saved best model on the test set once to report final performance.

The training set is used to fit the model, the validation set is used to decide when to stop, and the test set is used only for final evaluation. The test set should not be used to make stopping decisions, because that would leak information and bias the final estimate of performance.

---

### Question 8: Regularization  (10 points)

Briefly explain a few common methods of regularization to prevent overfitting.

---
**Answer:** 

Common regularization methods include:

- *Dropout*: randomly disables some neurons during training so the network does not rely too much on any one pathway.
- *Early stopping*: stops training when validation performance stops improving, which helps prevent overfitting.
- *Data augmentation*: creates modified training examples to increase the diversity of the dataset and improve generalization.

---